# Cvičenie 4 - Feed-forward neural network

Na tomto cvičení uviditíte, ako je možné vytvoriť jednoduchú feed-forward neurónovu sieť, slúžiacu na klasifikáciu obrázkov s ručne nakreslenými číslicami. Úloha najprv krok po kroku ukazuje jednotlivé kroky jej riešenia až po úspešné vytvorenie, natrénovanie a otestovanie siete. Po preštudovaní budete mať priestor získané znalosti aplikovať na implementácii vašej vlastnej neurónovej siete.

## Ciele

1. Oboznámiť sa s konceptom feed-forward neurónových sití
2. Krok po kroku si predstaviť proces implementácie FF neurónovej siete
3. Získané znalosti aplikovať na praktickej úlohe
4. Navrhnúť, implementovať a otestovať vlastnú FF neurónová sieť

## Zadanie

Detailne preskúmajte implementovanú vzorovú úlohu, ktorá spočíva vo vytváraní neurónovej siete určenej na klasifikáciu obrázkov rukou písaných číslic. Na základe nej krok po kroku potom implementujte vlastnú neurónovú sieť, ktorej úlohou je klasifikovať obrázkové dáta z dátovej sady CIFAR10, natrénujte a otestujte ju.

## Krok 1 - riešená vzorová úloha 

V tejto polovici cvičenia si najprv pozorne preštudujte vyriešenú vzorovú úlohu. Konkrétne ide o implementáciu neurónovej siete určenej na klasifikáciu rukou písaných číslic.

### 1.1 Importy

Na začiatok si naimportujeme potrebné knižnice. V našom prípade ide o knižinu Pytorch a jej balíky, ktoré budeme využívať plus knižnicu Matplotlib, slúžiacu na vizualizáciu dát. 

##### Spustiteľná ukážka

In [ ]:
import torch # pytorch
import torch.nn as nn # pytorch neural network module
import torch.nn.functional as F # pytorch neural network module
import torchvision # pytoch computer vision module
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
from tqdm import tqdm
from datetime import timedelta
import time

### 1.2 CPU / GPU ?

V tomto kroku si zvolíme, na aké zariadenie chceme využívať pri trénovaní/testovaní našej siete. Ak má váš počítač k dispozícií CUDA-enabled grafický procesor (napr. grafické karty od Nvidie), vužiejme ho na dosiahnutie značného zredukovania doby trénovia, resp. testovania. Na zistenie, či vaše zariadnie obsahuje CUDA-enabled procesor slúži v Pytorch-i funkcia *cuda.is_available()*

##### Spustiteľná ukážka

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Tvoj počítač podporuje CUDA, použijem GPU")
else:
    device = torch.device('cpu')
    print("Tvoj počítač nepodporuje CUDA, použijem CPU")

### 1.3 Vytvorenie Dataset-u, Dataloader-u

V tomto kroku vytvoríme objekty tried Dataset a Dataloader, ktoré slúžia na načítanie naších trénovacích resp. testovacích dát a ich následné dávkovanie do siete počas procesu trénovania a testovnia. V našom prípade použijeme dataset MNIST, ktorý obsahuje 60 000 trénovacích a 10 000 testovacích obrázkov ručne písaných císlic. Tento dataset je ako jeden z mnohých dostupný v balíku *torchvision*, ponúkanom v knižnici *pytorch*.

Premenná *batch_size* je hyperparameter, ktorý určije objem dát, ktorý vstupuje do siete v jednom kroku tŕenovania.

##### Spustiteľná ukážka

In [ ]:
batch_size = 100 # hyperparameter, 100 for now

# dataset with training data
train_dataset = torchvision.datasets.MNIST( # we use the MNIST object from torchvisien datasets, change to other one when using different dataset
    root='./Data', # path to store our data
    download=True, # download data locally, to specified path
    train=True, # take data from MNIST training set (to ensure we never evaluate on training data)
    transform=transforms.ToTensor(),  # transforms to be applied to all samples (in our case, fransform the data into Pyorch tensor)
)

# dataset with testing data
test_dataset = torchvision.datasets.MNIST( # we use the MNIST object from torchvisien datasets, change to other one when using different dataset
    root='./Data', # path to store our data
    train=False, # take data from MNIST testing set (to ensure we never evaluate on training data)
    transform=transforms.ToTensor() # transforms to be applied to all samples (in our case, fransform the data into Pyorch tensor)
)

# dataloader for training data
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset, # what dataset to load from (our MNIST)
    batch_size=batch_size, # number of samples used in one training iteration
    shuffle=True # shuffle the samples
)

# dataloader for testing data
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset, # what dataset to load from (our MNIST)
    batch_size=batch_size,  # number of samples used in one training iteration
    shuffle=False # dont the samples (this way, we always evaluate on the same data)
)

Tu môžte vidieť príklad dát z MNIST datasetu. Ako vidíte, ide o obrázky o veľkosti 28x28 pixelov, zobrazujúce biele ručne písané číslice na čiernom pozadí. 

##### Spustiteľná ukážka

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))

for i in range(2):
    for j in range(2):
        img, label = test_dataset[2*i + j]
        axes[i, j].imshow(img[0], cmap="Greys_r")
        axes[i, j].set_title(f"Číslica {label}")

### 1.4 Vytvorenie feed-forward neurónovej siete

V tomto kroku sa dostávame k vytvoreniu samotnej neurónovej siete. Ako je zo špecifikácie úlohy zjavné, ide o jednoduchú feed-forward neurónovú sieť. Sieť má v našom prípade nasledovnú štruktúru:
1) **Vstupná vrstva** - o veľkosti 784 neurónovm pretože naše obrázky majú tvar 28 x 28 px, teda jeden neurón pre každý pixel
2) **Jedna skrytá vrstva** - o veľkosti 100 neurónov. Toto je nami ľubovoľné číslo, je možné s ním experimentovať a sledovať, ako sa mení výkon siete
3) **Výstupná vrsta** - o veľkosti 10 neurónov. Táto veľkosť je daná tým, že pri klasifikácií máme 10 možných tried (10 číslic), ktoré môžme priradiť obrázkom. Všeobecne teda platí, že veľkosť výstupnej vrstvy siete pri klasifikačných problémoch záleži od počtu možných tried, ktoré môžme dátam priradiť.

##### Spustiteľná ukážka

In [ ]:
# hyperparameters of the network, these are given from the stand and don't change during training
input_size = 28 * 28 # size of the input layer, derived from size of the MNIST images (28x28 pixels)
hidden_size = 100 # size of our hidden layer, can be tweaked to achieve better performance 
num_classes = 10 # size of the output layers, equals the number of classes in the classification (10 digits)

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()

        self.fc_hidden = nn.Linear(input_size, hidden_size)  # hidden layer object
        self.fc_output = nn.Linear(hidden_size, num_classes) # output layer object       
    
    def forward(self, x):
        out = F.relu(self.fc_hidden(x)) # first pass the input data through the hidden layer, than into a Relu function
        out = self.fc_output(out) # pass the data into the output layer, to get a predicted value for each class (each digit)
        return out 
    
model = NeuralNet(input_size, hidden_size, num_classes).to(device) # initialize a new network with our hyperparameters (on our specified devide, to utilize GPU if available)

V tomto bode naša sieť ešte nie je natrénovaná, takže predikcie, ktoré poskytuje pravdepodobne nebudú mať veľkú hodnotu. Ukážeme si to na príklade konkrétneho obrázka.

##### Spustiteľná ukážka

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))

for i in range(2):
    for j in range(2):
        img, label = test_dataset[2*i + j] # image to predict on
        prediction = model(img[0].flatten().to(device)) # pass image flattened image into the network to get class predictions
        predicted_label = torch.argmax(prediction) # the class with the highest values
        
        axes[i, j].imshow(img[0], cmap="Greys_r")
        axes[i, j].set_title(f"Predpoveď siete = {predicted_label}")


Ako môžete vidieť, sieť nepredpovedala správnu triedu ani pre jeden z obrázkov. Našou ďalšou úlohou bude teda ju natrénovať, aby bola reálne schopná zistiť hodnotu číslic na obrázkoch.

### 1.5 Trénovanie siete

Už máme vytvorenú jednoduchú sieť no ako sme si ukázali, sama o sebe nie je schopná plniť svoj účel. Ďalším krokom teda je natrénovať na trénovacích dátach. Trénovanie prebieha v tzv. epochách, tj. počet iterácií, koľko krát je sieť trénovaná na trénovacích dátach. Navyše vďaka dataloaderu, ktorý sme si vytvorili už skôr, nemusíme trénovacie dáta do sieťe dávkovať po jednom (čo by bolo časovo náročné), ale v dávkach o nami zvolenej veľkosti (batch_size, v našom prípade 100).

Trénovacia slučka má následne takýto proces:
1) Zvoliť si loss function, tj. funkciu, ktorá určuje tzv. "stratu", teda na koľko sa hodnoty predpovedané sieťou líšia od "správnych" tried
2) Zvoliť si optimizačný algoritmus, ktorého úlohou je nastaviť váhy v sieti tak, aby predpovede generované sieťou vykazovali menšiu stratu, pomocou metódy gradient descent
3) Pre každú epochu trénovania:<br/>
    3.1) Pre každú dávku dát (nami určenej veľkosti):<br/>
        3.1.1) Prípadne upraviť tvar dát - aby sa zaistilo, že dáta sú do siete zadávané v spŕavnom tvar. V našom prípade konvertovať dávku 2d tensorov obrázkov na dávku 1d vyrovnaných tensorov (keďže sieť požaduje ploché dáta)<br/>
        3.1.2) Vykonať popredný prechod - získať predpovedané triedy pre dáta v dávke<br/>
        3.1.3) Vynulovať gradienty optimizátora - aby sa gradienty medzi dávkami nesčítavali<br/>
        3.1.4) Vypočítať stratu - na základe predpovedaných a reálnych tried dát, pomocou zvolenej loss function<br/>
        3.1.5) Vykonať spätný prechod - na základe straty spatne vypočítať gradienty<br/>
        3.1.6) Upraviť váhy v sieti pomocou optimizátora - na základe gradientov vypočítaných pri spätnom prechode, tak aby budúce predpovede vykazovali menšiu stratu<br/>
Nasledujúci kód iteratívne vykonáva všetky hore uvedené kroky a priebežne loguje informácie, aby bolo možné vidieť ako sa mení vypočítaná strata počas trénovania:

##### Spustiteľná ukážka

In [ ]:
model = NeuralNet(input_size, hidden_size, num_classes).to(device)

# training hyperparamters
num_epochs = 10
learning_rate = 0.001

criterion = nn.CrossEntropyLoss() # the loss function we'll use for training
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # the optimizing algorithm

start_time = time.time()
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1} training progress:")

    for i, (images, labels) in enumerate(tqdm(train_loader)): 
        images = images.reshape(-1, 28*28).to(device) # reshape data accordingly
        labels = labels.to(device)
        
        outputs = model(images) # forward pass - do a prediction on our batch

        optimizer.zero_grad() # reset the gradients (so that they don't ad up between batches)
                
        loss = criterion(outputs, labels) # calculate loss - the difference between predicted and actual labels
        loss.backward() # backward pass - backpropagate the loss, calculating gradients for neurons
        
        optimizer.step() # let the optimiser adjust weights based on calculated gradients - to minimise loss
    
    if epoch + 1 == num_epochs: 
        print (f'\nTraining took {str(timedelta(seconds=time.time() - start_time))}, loss after all epochs: {loss.item():.5f}\n')    
    else:
        print (f'Loss at end of epoch {epoch+1}: {loss.item():.5f}\n')

Ako môžte vidieť, vypočítaná strata časom klesala, teda je zjavné, že trénovvanie splnilo svoj účel. Ako neskôr uvidíte, presnosť siete sa naozaj trénvaním zvýšila.

### 1.6 Testovanie siete

Keď sme už sieť natrénovali, ostáva nám už len ju otestovať, aby sme zistili nakoľko bolo trénovanie úspešné. Spôsobov ako to vykonať je mnoho, no pri klasifikačných úlohách je prirodzenou možnosťou vykonať predikciu na veľkom množstve testvacích dát a zisiť, na koľkých vzorkach sieť predpovedala správnu triedu. Opäť nám pri tom pomôže trieda Dataloader, ktorý nám umožňuje vykonávať predpoveď na balíku dát naraz.

##### Spustiteľná ukážka

In [ ]:
with torch.no_grad(): # don't adjust gradients (to reduce memory usage)   
    correct_count = 0
    total_count = 0

    # for each batch, make predictions and calculate how many of them were correct
    for images, labels in tqdm(test_loader):
        images = images.reshape(-1, 28*28).to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted_labels = torch.max(outputs.data, 1)
        
        total_count += labels.size(0)
        correct_count += (predicted_labels == labels).sum()
        
accuracy = 100. * correct_count / total_count
print (f'Messured accuraccy: {accuracy:0.3f}% \n')

### 1.7. Záver

Keď už máme sieť úspešne natrénovanú a otestovanú, môžme ju otestovať na vlastných obrázkoch. Nastav hodnotu premennej *my_path* na cestu k obrázku v tvojom počítači a otestuj, ako ho sieť klasifikuje

**<font color='red'>Pozor, keďže sieť bola trénovaná na štvorcových obrázkoch s bielym textom na čiernom pozadí, najlepšie výsledky dosiahne, keď váš obrázok bude bude mať tiež takú formu.</font>**

##### Spustiteľná ukážka

In [ ]:
from PIL import Image

my_path = "./example_img.png"

img = Image.open(my_path)

preprocess = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((28,28)),
    transforms.ToTensor()
])

img = preprocess(img)

prediction = model(img[0].flatten().to(device))
predicted_label = torch.argmax(prediction)

plt.title(f"Predpovedaná trieda = {predicted_label}")
plt.imshow(img[0], cmap="Greys_r")

## Krok 2 - implementácia vlastnej FF siete

Ak ste si prešli vyššie ukázanú implementáciu, v tejto polovici cvičenia si môžte vaše nadobudnuté znalosti prakticky vyskúšať. Postupujte v rovnakých krokoch do akých bola delená vzorová úloha a navrhnite vlastnú neurónoú sieť. Opäť budete pracovať s balíkom **torchvison** a vašou úohou bude implementovať neurónovú sieť určenú na klasifikáciu obrázkov poskytovaných datasetom **CIFAR10**.

### Úloha č.1:

Naimportujte si potrebné knižnice pre implementáciu siete.

In [ ]:
# your necessary imports here

### Úloha č.2:

Otestujte či vaše zariadenie obsahuje CUDA-enabled GPU, ak áno použite ho. Dplňte nasledujúci fragment kódu.

In [ ]:
if ___: # test if cuda is available
    # CUDA available
else:
    # CUDA not available

### Úloha č.3:

Získajte dataset **CIFAR10** ponúkaný balíkom torchvision. Preskúmajte jeho štruktúru a formu dát ktoré obsahuje.

Dokumentáciu k datasetu: https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html#torchvision.datasets.CIFAR10

In [ ]:
batch_size = # define your batch size

# dataset with training data
train_dataset = torchvision.datasets.CIFAR10(
    # add dataset parameters
)

# dataset with testing data
test_dataset = torchvision.datasets.CIFAR10(
    # add dataset parameters
)

# dataloader for training data
train_loader = torch.utils.data.DataLoader(
    # add dataset parameters
)

# dataloader for testing data
test_loader = torch.utils.data.DataLoader(
    # add dataset parameters
)

**Doplňujúca úloha:**
Pomocou knižnice matplotlib sa pokúste vizualizovať dáta v datasete

In [ ]:
# your code here

### Úloha č.4:

Vytvorte triedu vašej siete. Nezabudnite v nej implementovať **konštruktor** a metódu **forward**, obdobne ako tomu bolo vo vzorovej úlohe. Experimentujte pri tom so štruktúrou siete (skrytých vrstiev). 

**Nezabudnite správne určiť hyperparametre siete (na veľkosti vstupných obrázkov a počtu tried)**

In [ ]:
# correctly assign these hyper parameters according to the dataset, add ones for hidden layers
input_size = 
num_classes = 

class MyNet(nn.Module):
    def __init__(self, _ ): # add input parameters
        super(MyNet, self).__init__()

        # define your layers here  
    
    def forward(self, x):
        # define your forward pass
        
        return out 

### Úloha č.5:

Keď ste si navrhli model vašej siete, doplňte nasledujúci kód a implementujte tak trénovaciu slučku siete. Vychádzajte zo štruktúry slučky ukázanej v kroku 1.5. Kód doplňte o hyperparamter, vytvorenie stratovej funkcie, optimizátora a krok vykonania predikcie, vypočítania stratovej funkcie a vykonanie spätného kroku. 

In [ ]:
model = _ # create your model (MyNet class)

# assign training hyperparamters
num_epochs = _
learning_rate = _ 

# choose what loss function and optimiser you want to use
criterion = _
optimizer = _

for epoch in range(num_epochs):
    for i, (batch_X, batch_Y) in enumerate(tqdm(train_loader)): 
        images = images.reshape(-1, 32 * 32 * 3).to(device)
                
        # make prediction on batch (forwardpass)
        outputs = _

        optimizer.zero_grad()
        # calculate loss, do backward pass and make an optimiser step
        

#### Úloha na preskúšanie č.6:

Po úspešnom natrénovaní siete implementuje trénovaciu slučku siete. Opäť vychádzajte zo slučky ukázanej v kroku 1.6, ktorá vypočítava percento správne ozačkovaných testovacích dát.

In [ ]:
with torch.no_grad(): # don't adjust gradients (to reduce memory usage)   
    correct_count = 0
    total_count = 0

    # for each batch, make predictions and calculate how many of them were correct
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = _ # make prediction
        
        # calculate the amount of correct predictions
        
accuracy = 100. * correct_count / total_count
print (f'Messured accuraccy: {accuracy:0.3f}% \n')